In [3]:
import pandas as pd

# This will work for anyone who downloads your GitHub repo, as long as they put the CSV in the same folder
df = pd.read_csv('sentiment_data.csv')

print("Data loaded successfully!")
print(df.head())
print(df.columns)

Data loaded successfully!
                                 Sentence Sentiment
0     The service was excellent and fast.  positive
1       What a wonderful day it is today.  positive
2    This is the best meal I've ever had.  positive
3          I'm so happy with the results.  positive
4  The movie was fantastic and inspiring.  positive
Index(['Sentence', 'Sentiment'], dtype='object')


In [4]:
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Clean the text (lowercase and remove punctuation)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Keep only letters and spaces
    return text

# Create a new column for the cleaned text
df['clean_sentence'] = df['Sentence'].apply(clean_text)

# 2. Define Features (X) and Labels (y)
X = df['clean_sentence']
y = df['Sentiment']

# 3. Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Convert text to numbers using TF-IDF Vectorizer
# max_features=1000 is good for a 600-sentence dataset
vectorizer = TfidfVectorizer(max_features=1000)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Data split and vectorized successfully!")
print(f"Training set size: {X_train_vec.shape[0]} sentences")
print(f"Testing set size: {X_test_vec.shape[0]} sentences")

Data split and vectorized successfully!
Training set size: 486 sentences
Testing set size: 122 sentences


In [5]:
from sklearn.linear_model import LogisticRegression

# Initialize the model (max_iter=1000 ensures it has enough time to learn)
model = LogisticRegression(max_iter=1000)

# Train the model using the training data
model.fit(X_train_vec, y_train)

print("Model training complete!")

Model training complete!


In [6]:
from sklearn.metrics import classification_report, accuracy_score

# Make predictions on the test data
y_pred = model.predict(X_test_vec)

# Print accuracy score
print("Accuracy Score:", accuracy_score(y_test, y_pred))

# Print detailed classification report (Precision, Recall, F1-Score)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy Score: 0.6229508196721312

Classification Report:

              precision    recall  f1-score   support

    negative       0.58      0.41      0.48        46
     neutral       0.74      0.97      0.84        32
    positive       0.55      0.59      0.57        44

    accuracy                           0.62       122
   macro avg       0.62      0.66      0.63       122
weighted avg       0.61      0.62      0.61       122



In [7]:
# Add your own sentences here to test!
new_sentences = [
    "I absolutely love this product, it works perfectly.",
    "This is the worst experience I have ever had.",
    "The package arrived today. It is okay.",
    "I am so happy with the service!"
]

# Clean and vectorize the new sentences using the same rules as before
new_clean = [clean_text(s) for s in new_sentences]
new_vec = vectorizer.transform(new_clean)

# Predict
predictions = model.predict(new_vec)

# Display the results
for sentence, prediction in zip(new_sentences, predictions):
    print(f"Sentence: '{sentence}'")
    print(f"Predicted Sentiment: -> {prediction}\n")

Sentence: 'I absolutely love this product, it works perfectly.'
Predicted Sentiment: -> positive

Sentence: 'This is the worst experience I have ever had.'
Predicted Sentiment: -> negative

Sentence: 'The package arrived today. It is okay.'
Predicted Sentiment: -> neutral

Sentence: 'I am so happy with the service!'
Predicted Sentiment: -> positive

